In [1]:
import pandas as pd

df = pd.read_csv('diabetic_data.csv')

df.head()          # 실제 값 생김새 보기
df.info()          # 컬럼별 자료형, 결측 여부(non-null count) 한눈에
df.describe()      # 숫자형 컬럼만 요약 (평균, 최소/최대 등)

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


전부 non-null로 나왔지만 결측지가 없는게 아니라 이 데이터를 살펴보면 결측값을 빈칸이 아니라 '?'로 표시해뒀음 문자열로 인식해서 non-null이 나온거 같음 실제로 결측이 얼마나 있는지 ?를 nan으로 바꿔서 확인 필요
0값들은 결측치가 아니라 진짜 0 / number_emergency(응급실 방문 횟수) 같은 건 "0번 방문"이 실제로 있을 수 있는 정상적인 값

In [2]:
import numpy as np

df = df.replace('?', np.nan)
df.isnull().sum()


encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

결측값 있는 열들 비율 확인

In [3]:
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_pct[missing_pct > 0].sort_values(ascending=False)

weight               96.9
max_glu_serum        94.7
A1Cresult            83.3
medical_specialty    49.1
payer_code           39.6
race                  2.2
diag_3                1.4
diag_2                0.4
dtype: float64

weight(체중 — 대부분 미기록이라 결측 많음) = 결측 너무 많아서 제외
max_glu_serum (혈당 검사 결과 — 검사 안 하면 결측) = 결측 너무 많아서 제외
A1Cresult (당화혈색소 검사 결과 — 검사 안 하면 결측, 이번 분석 핵심 변수)
medical_specialty (담당의 전문분야) = 결측도 많고 질문에 상관없음, 제외
payer_code (진료비 지불방식 코드) = 결측도 많고 질문에 상관없음, 제외
race (인종) = 유지, 결측은 "Unknown"으로 / 결측 비율 낮고 분석적으로 의미있음/
 인종에 따라 검사 받는 비율이 다른가? 알아볼수있음 / 5단계 심화(세그먼트 분석)나 6단계 한계 섹션("교란요인으로 인종·의료접근성 차이가 있을 수 있다")에서 쓸 소재
diag_2 (이차 진단명), diag_3 (세 번째 진단명) = 결측은 거의 없지만 형식때문에 안 씀, ICD-9 진단 코드가 숫자로 돼 있어서 별도의 ICD-9 코드북과 매핑하는 작업이 필요 / 이번에는 분석 범위 밖

A1Cresult의 NaN은 데이터 손실이 아니라 "검사 미실시"를 의미하는 유효한 범주 (공식 문서 기준 결측값 없음). pandas가 원본의 문자열 'None'을 자동으로 NaN 처리한 것뿐이므로, 그대로 '검사 안함' 그룹으로 사용

In [4]:
cols_to_drop = ['weight', 'max_glu_serum', 'medical_specialty', 'payer_code', 'diag_1', 'diag_2', 'diag_3']

before_cols = df.shape[1]
df = df.drop(columns=cols_to_drop)
after_cols = df.shape[1]

print(f'정제 전 컬럼 수: {before_cols} -> 정제 후 컬럼 수: {after_cols}')

정제 전 컬럼 수: 50 -> 정제 후 컬럼 수: 43


In [5]:
df['race'] = df['race'].fillna('Unknown')

# 확인
df['race'].value_counts(dropna=False)

race
Caucasian          76099
AfricanAmerican    19210
Unknown             2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

In [6]:
df['tested_A1C'] = df['A1Cresult'].notna().map({True: '검사함', False: '검사안함'})
df['tested_A1C'].value_counts()

tested_A1C
검사안함    84748
검사함     17018
Name: count, dtype: int64

사망 환자는 재입원 분석 대상에서 제외 / discharge_disposition_id 값 중에 11, 19, 20, 21번은 "Expired"(사망)을 뜻함 사망환자는 재입원 불가, 그래도 재입원 여부에 no로 표시 => 구분 필요

In [7]:
expired_codes = [11, 19, 20, 21]
n_expired = df['discharge_disposition_id'].isin(expired_codes).sum()
print(f'사망(Expired)으로 퇴원 처리된 건수: {n_expired}')

사망(Expired)으로 퇴원 처리된 건수: 1652


In [8]:
expired_codes = [11, 19, 20, 21]

before = len(df)
df = df[~df['discharge_disposition_id'].isin(expired_codes)]
after = len(df)

print(f'정제 전: {before}행 -> 정제 후: {after}행 (제거: {before - after}건)')

정제 전: 101766행 -> 정제 후: 100114행 (제거: 1652건)


"사망(discharge_disposition_id 11, 19, 20, 21)으로 퇴원한 1,652건은 재입원이 구조적으로 불가능하므로 분석에서 제외함. 정제 전 101,766행 → 정제 후 100,114행."

중복확인 / encounter_id (입원 건 ID) = 입원할때마다 환자한테 부여되는 번호 / patient_nbr (환자 ID) = 환자 자체 ID(주민번호같은형식)

In [9]:
full_dup = df.duplicated().sum()
print(f'전체 행 기준 완전 중복: {full_dup}')

전체 행 기준 완전 중복: 0


In [10]:
dup_encounters = df['encounter_id'].duplicated().sum()
dup_patients = df['patient_nbr'].duplicated().sum()

print(f'encounter_id 중복: {dup_encounters}')
print(f'patient_nbr 중복(동일 환자 여러 입원): {dup_patients}')

encounter_id 중복: 0
patient_nbr 중복(동일 환자 여러 입원): 29675


동일 환자의 반복 입원 기록은 서로 독립적이지 않아 검정의 독립성 가정을 위반하므로 환자당 첫 입원 기록만 남겨 분석 단위를 '환자'로 통일함

In [11]:
before = len(df)
df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')
after = len(df)

print(f'정제 전: {before}행 -> 정제 후: {after}행 (제거: {before - after}건)')

정제 전: 100114행 -> 정제 후: 70439행 (제거: 29675건)


배정일관성/ 컬럼에 들어있는 값들이 정의된 범주 안에 들어오는지 확인

In [12]:
print(df['gender'].unique())
print(df['readmitted'].unique())
print(df['age'].unique())
print(df['A1Cresult'].unique())

<ArrowStringArray>
['Female', 'Male', 'Unknown/Invalid']
Length: 3, dtype: str
<ArrowStringArray>
['NO', '>30', '<30']
Length: 3, dtype: str
<ArrowStringArray>
[ '[80-90)', '[90-100)',  '[40-50)',  '[50-60)',  '[60-70)',  '[70-80)',
  '[20-30)',  '[10-20)',  '[30-40)',   '[0-10)']
Length: 10, dtype: str
<ArrowStringArray>
[nan, '>7', '>8', 'Norm']
Length: 4, dtype: str


GENDER에 'Unknown/Invalid' 이상 값이 있다는 걸 확인 / 이번 분석에서는 gender를 직접 안 쓰니까 굳이 수정 안 함 

In [13]:
n_tested = df['A1Cresult'].notna().sum()
n_not_tested = df['A1Cresult'].isna().sum()

print(f'A1C 검사함: {n_tested}')
print(f'A1C 검사 안함: {n_not_tested}')

A1C 검사함: 12878
A1C 검사 안함: 57561


A1C 검사함/검사안함 비율(약 18:82)은 SRM 위반이 아니라 애초에 무작위 배정이 아닌 관찰 데이터의 특성 이는 이후 인과 해석 시 반드시 감안해야 할 핵심 한계

In [14]:
from scipy import stats
n_tested, n_not_tested = 12878, 57561
stats.chisquare([n_tested, n_not_tested], [(n_tested + n_not_tested) / 2] * 2)

Power_divergenceResult(statistic=np.float64(28344.67395902838), pvalue=np.float64(0.0))

형식적으로 카이제곱검정을 돌리면 p ≈ 0으로 나오지만 이는 SRM(배정 로직 버그)의 증거가 아님. 애초에 A1C 검사 여부가 50:50 무작위 배정이 아니라 의료진 판단에 따른 것이므로 두 그룹 크기가 크게 다른 것은 자연스러운 현상. 이 비대칭 자체가 이후 인과 해석에서 반드시 감안해야 할 한계.

질문
HbA1c(A1C) 검사를 받은 환자가 검사를 받지 않은 환자보다 30일 이내 재입원율이 낮은가?

귀무가설
: A1C 검사 여부와 30일 이내 재입원율은 관계가 없다 (검사함 재입원율 = 검사안함 재입원율)

대립가설
: A1C 검사를 받은 환자의 30일 이내 재입원율이 검사를 받지 않은 환자보다 낮다 (검사함 재입원율 < 검사안함 재입원율)

유의수준과 방향

𝛼 = 0.05
단측검정 — 방향을 미리 "검사함 쪽이 더 낮다"로 고정. (근거: 원 논문 Strack et al.의 연구 가설과 같은 방향으로 미리 정하는 것 — 데이터를 보고 나서 방향을 정하면 p-해킹이 되므로, 지금 이 시점에 방향을 확정)

결과 지표 정의
readmitted 컬럼의 3개 값(NO, <30, >30) 중 <30을 "재입원함(1)", 나머지(NO, >30)를 "재입원 안함(0)"으로 이진화. (30일 이내 재입원이 임상적으로 가장 중요하게 보는 지표이기 때문 — 미국 CMS 재입원 프로그램 기준도 30일 기준을 씀)

In [15]:

df['readmit_30'] = (df['readmitted'] == '<30').astype(int)
df['tested_A1C'] = df['A1Cresult'].notna().astype(int)   # 1=검사함, 0=검사안함

df.groupby('tested_A1C')['readmit_30'].mean()

tested_A1C
0    0.090617
1    0.083864
Name: readmit_30, dtype: float64

이번 분석은 이진 범주형이고 비교하려는 집단도 A1C 검사함/검사안함 2개뿐, 결과 변수가 이진 범주형이므로 정규성·등분산 검정(Shapiro-Wilk, Levene)은 해당하지 않음. 대신 각 그룹의 성공/실패 빈도가 모두 충분히 커서(5 이상) 정규근사가 안정적이라는 조건을 확인. 카이제곱 검정과 두비율 검정 적용 가능 / 지금까지 배웠던 카이제곱 검정을 최종적으로 선택, 다만 카이제곱 자체가  양측검정이므로 사전에 정한 단측 가설에 맞춰 양측 P값을 2로 나누어 단측 P값으로 진행

In [16]:
import pandas as pd
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['tested_A1C'], df['readmit_30'])
print(contingency)

chi2, p_two_sided, dof, expected = chi2_contingency(contingency)
print(f'chi2 = {chi2:.3f}, p(양측) = {p_two_sided:.4f}')

p_one_sided = p_two_sided / 2
print(f'p(단측, 방향 일치 확인됨) = {p_one_sided:.4f}')

readmit_30      0     1
tested_A1C             
0           52345  5216
1           11798  1080
chi2 = 5.813, p(양측) = 0.0159
p(단측, 방향 일치 확인됨) = 0.0080


In [17]:
import numpy as np

n1 = 11798 + 1080   # 검사함 표본수
n0 = 52345 + 5216   # 검사안함 표본수
x1 = 1080            # 검사함 재입원 수
x0 = 5216            # 검사안함 재입원 수

p1 = x1 / n1
p0 = x0 / n0

abs_diff = p1 - p0
rel_diff = p1 / p0

se = np.sqrt(p1*(1-p1)/n1 + p0*(1-p0)/n0)
ci_low = abs_diff - 1.96 * se
ci_upp = abs_diff + 1.96 * se

n_total = n1 + n0
cramers_v = np.sqrt(chi2 / n_total)

print(f'검사함 재입원율: {p1:.4f} ({p1*100:.2f}%)')
print(f'검사안함 재입원율: {p0:.4f} ({p0*100:.2f}%)')
print(f'절대 차이: {abs_diff*100:.2f}%p')
print(f'상대 차이: {rel_diff:.2f}배')
print(f'95% CI (차이): [{ci_low*100:.2f}%p, {ci_upp*100:.2f}%p]')
print(f"Cramér's V = {cramers_v:.4f}")

검사함 재입원율: 0.0839 (8.39%)
검사안함 재입원율: 0.0906 (9.06%)
절대 차이: -0.68%p
상대 차이: 0.93배
95% CI (차이): [-1.21%p, -0.14%p]
Cramér's V = 0.0091


A1C 검사를 받은 환자의 30일 이내 재입원율이 검사를 받지 않은 환자보다 낮았다 (8.39% vs 9.06%).
-0.68%p, 상대로는 0.93배(약 7% 낮음), χ²(1) = 5.81, p = .008(단측), 95% CI [-1.21%p, -0.14%p], Cramér's V = 0.009

해석 : 통계적으로는 유의미한 차이다 (P=0.008 <0.05) 하지만 효과 크기(Cramér's V = 0.009)는 일반적 기준(0.1 미만)으로 볼때 사실상 무시할 수 있는 수준이며, 절대차이도 0.68%P에 불과함. 이는 표본이 매우 크기때문에(약 7만명) 실질적으로 미미한 차이도 통계적으로 유의미하게 나타난 경우로 보임. 즉 "A1C 검사가 재입원을 확실히 줄인다"고 강하게 주장하기는 어렵고, "차이가 있긴 하지만 그 크기는 작다"는 게 더 정확한 결론. 

카이제곱 통계량과 p값은 "이 차이가 우연일 가능성이 얼마나 낮은가"를 나타내는 확신의 정도이며, 표본 크기의 영향을 크게 받음 — 표본이 크면 아주 작은 차이도 통계적으로 유의하게 나올 수 있음

반면 효과크기(Cramér's V)는 표본 크기와 무관하게 "그 차이가 실제로 얼마나 큰가"를 나타냄

이번 분석에서는 χ²(1)=5.81, p=.008로 통계적으로 유의했지만, Cramér's V=0.009로 효과크기는 무시할 수 있는 수준이었다. 이는 표본이 매우 커서(약 7만 명) 실질적으로 미미한 차이도 유의하게 나타난 사례이며, "통계적 유의성"과 "실질적 중요성"은 다르다는 것을 보여준다.

95% CI (차이): [-1.21%p, -0.14%p] / "절대 차이(-0.68%p)"에 대한 95% 신뢰구간
의미: "우리가 관측한 -0.68%p라는 차이는 하나의 추정치일 뿐이고, 진짜 모집단의 차이는 (95% 확신으로) -1.21%p에서 -0.14%p 사이 어딘가에 있을 것이다"라는 뜻
중요한 포인트: 이 구간이 0을 포함하지 않음 (전부 음수). 이건 "차이가 없다(0)"는 가능성을 배제할 수 있다는 뜻이라, p<.05로 유의하다는 결과와 정확히 같은 얘기를 다른 방식으로 보여주는 거예요.(바꿔 말하면: CI가 0을 안 낀다 = "차이가 없다"는 시나리오가 그럴듯한 후보 범위에서 빠졌다 = H0를 기각할 근거가 있다(유의함). p값이 "숫자 하나로 유의성만 알려주는 것"이라면, CI는 "그 유의성 판단이 나온 이유를 범위로 직접 보여주는 것"이라고 볼 수 있어요.)
반대로 구간의 폭(-1.21 ~ -0.14)이 좁다는 건, 표본이 커서 추정이 꽤 정밀하다는 뜻이에요.(여기서 n1, n0(표본 크기)이 분모에 있어요. 분모가 커지면(표본이 많아지면) se는 작아져요. 그리고 CI 폭은 ± 1.96 × se로 정해지니까, se가 작아지면 CI 폭도 좁아져요.

직관적 비유: 여론조사에서 100명한테 물어본 결과랑, 10만 명한테 물어본 결과를 비교하면 — 100명 조사는 "지지율 45% ± 10%p" 처럼 범위가 넓고, 10만 명 조사는 "45% ± 0.3%p"처럼 범위가 훨씬 좁아지죠. 사람이 많을수록 우연히 튀는 값들이 서로 상쇄돼서, 진짜 값에 더 가깝게 좁혀지는 거예요.)


5단계 A. 이 실험은 효과를 잡을 ’힘’이 있었나

4단계 결과 보면 통계적으로 유의하지만 효과크기는 미미하다는 결과가 나옴 => 이게 정말 작은 효과를 정확히 잡아낸건가, 아니면 우연히 걸린건가 라는 의문 생김 (B는 정규성 문제인데 우린 비율 데이터라 해당 없고, C는 세그먼트 다중비교인데 우리 질문엔 세그먼트가 없고, D는 비용 계산인데 정보가 부족해서 A가 제일 자연스러움)

In [18]:
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

analysis = NormalIndPower()

# 관측된 효과크기 기준
h_observed = proportion_effectsize(p1, p0)
n_needed = analysis.solve_power(effect_size=h_observed, alpha=0.05, power=0.8, ratio=n0/n1, alternative='smaller')
n_needed = float(np.asarray(n_needed).item())
print(f"관측된 효과크기(0.68%p 차이) 기준, 80% 검정력에 필요한 검사함 그룹 표본수: {n_needed:.0f}명")

# 1%p 차이 기준 (abs() 빼고 부호 그대로 유지 — 방향 지정과 일치시켜야 함)
h_1pp = proportion_effectsize(p0 - 0.01, p0)
n_needed_1pp = analysis.solve_power(effect_size=h_1pp, alpha=0.05, power=0.8, ratio=n0/n1, alternative='smaller')
n_needed_1pp = float(np.asarray(n_needed_1pp).item())
print(f"1%p 차이를 80% 검정력으로 잡으려면 필요한 검사함 그룹 표본수: {n_needed_1pp:.0f}명")

관측된 효과크기(0.68%p 차이) 기준, 80% 검정력에 필요한 검사함 그룹 표본수: 13206명
1%p 차이를 80% 검정력으로 잡으려면 필요한 검사함 그룹 표본수: 5918명


0.68%p 차이(우리가 실제로 관측한 차이)를 80% 검정력으로 잡으려면 그룹당 약 13,206명 필요
1%p 차이(더 큰 효과)를 잡으려면 그룹당 약 5,918명만 있어도 됨 (효과가 클수록 적은 표본으로도 잡을 수 있음 — 당연한 방향)

실제 검사함 그룹 표본수는 12,878명. 13,206명보다 살짝 적은데도 유의한 결과가 나온 이유는 실제 관측된 차이(0.68%p)가 우연히 딱 경계선 근처였기 때문. 이 부분은 다음 MDE 계산에서 더 명확하게 설명돼요.

"얼마나 필요했나" (필요 표본수 역산)

"0.68%p 차이를 80% 검정력으로 잡으려면 몇 명 필요한가?" → 13,206명
"1%p 차이라면 몇 명 필요한가?" → 5,918명
효과가 클수록(1%p > 0.68%p) 적은 표본으로도 충분히 잡을 수 있다는 걸 보여줌 — 그물이 커야 작은 물고기를 잡을 수 있다는 논리와 같은 방향

In [19]:
!pip install statsmodels

In [20]:
mde_h = analysis.solve_power(effect_size=None, nobs1=n1, ratio=n0/n1, alpha=0.05, power=0.8, alternative='smaller')
mde_h = float(np.asarray(mde_h).item())
print(f"현재 표본(검사함 {n1}명)으로 80% 검정력에서 탐지 가능한 최소 효과크기(Cohen's h): {mde_h:.4f}")

현재 표본(검사함 12878명)으로 80% 검정력에서 탐지 가능한 최소 효과크기(Cohen's h): -0.0242


In [21]:
from scipy.optimize import brentq

def h_given_p1(p1_try):
    return proportion_effectsize(p1_try, p0) - mde_h

p1_mde = brentq(h_given_p1, 0.001, p0)
mde_pp = (p0 - p1_mde) * 100
print(f"절대 차이로 환산: 약 {mde_pp:.3f}%p 이상의 차이는 현재 표본으로 탐지 가능")

절대 차이로 환산: 약 0.684%p 이상의 차이는 현재 표본으로 탐지 가능


MDE (Minimum Detectable Effect, 최소 탐지 가능 효과)

방향을 거꾸로 뒤집은 질문이에요: "지금 내가 가진 표본(12,878명)으로 80% 검정력을 유지하려면, 최소 얼마나 큰 효과여야 잡을 수 있나?"
답: 약 0.684%p
즉 "내 그물로는 0.684%p보다 작은 물고기는 애초에 못 잡는다"는 뜻이에요

6️ 의사결정으로 마무리

1. 질문과 가설
HbA1c(A1C) 검사를 받은 환자가 받지 않은 환자보다 30일 이내 재입원율이 낮은가? 
H0 : 두 그룹의 재입원율은 같다 vs H1 : 검사함 그룹이 더 낮다 (단측, α=0.05, 사전 설정).

2. 데이터 위생 점검

원본 101,766행 → 사망 환자 1,652건 제거 → 불필요 컬럼 7개 제거(weight, max_glu_serum, medical_specialty, payer_code, diag_1~3) → 환자당 첫 입원만 남겨 중복 제거(29,675건) → 최종 약 70,439행
race 결측은 'Unknown'으로 채움, A1Cresult의 결측은 "검사 미실시"라는 유효한 값으로 그대로 유지
gender에 이상값(Unknown/Invalid) 존재 확인했으나 이번 분석 미사용
A1C 검사함/안함 비율(18:82)은 SRM이 아니라 무작위 배정이 아닌 관찰 데이터의 자연스러운 특성

3. 검정 선택 근거
결과 변수가 이진(재입원 여부), 비교 집단 2개(검사함/안함)라 카이제곱 독립성 검정을 선택. 두 비율 z-검정도 있지만 기존에 배운 도구 범위 내에서 검증 진행, 방향(단측)은 양측 p값을 2로 나눠 반영.

4. 결과
검사함 8.39% vs 검사안함 9.06%, 절대차이 -0.68%p, 상대차이 0.93배(약 7% 낮음), χ²(1)=5.81, p=.008(단측), 95% CI [-1.21%p, -0.14%p], Cramér's V=0.009. 통계적으로 유의하나 효과크기는 무시할 수 있는 수준. MDE 분석 결과, 현재 표본은 약 0.68%p 이상의 효과만 탐지 가능했고 실제 관측치도 그 경계선에 걸쳐 있어, 결과가 유의하지만 여유 있게 확실한 수준은 아님.

5. 의사결정 — 보류 (데이터 추가 수집 권고)
"모든 환자에게 A1C 검사 의무화"를 강하게 권고하기엔 근거가 약하다. 방향성(검사함 쪽이 재입원율 낮음)은 있으나 효과크기가 매우 작고 결과가 표본 탐지 한계선에 걸쳐 있어, 다른 연도·다른 병원 데이터로 재현되는지 추가 확인이 필요하다. 즉시 "도입"이나 "폐기" 대신, 추가 데이터로 재검증 후 결정하는 것이 안전하다.

6. 한계

-무작위 배정 실험이 아닌 관찰 데이터 — A1C 검사 여부는 의료진 판단에 따른 것이라, 검사받은 환자군과 안 받은 환자군은 애초에 중증도·건강상태가 다를 수 있음(교란요인, 이번 분석에서 통제 안 함)
-환자당 첫 입원 기록만 사용해 독립성은 확보했지만, 반복 입원 환자의 이후 경과는 반영 못 함
-인종(race)·성별 등 잠재적 교란변수를 통제하지 않은 단순 비교
-효과크기가 작고 표본 탐지 한계선에 걸린 결과라, 이 결론은 "인과적으로 확실하다"기보다 "상관관계 수준의 약한 신호"로 해석해야 함

출처: UCI Machine Learning Repository, "Diabetes 130-US hospitals for years 1999-2008" (archive.ics.uci.edu). 다운로드일: 2026-07-31